# Statistics for Data Analysts/Engineers — Module 3: Probability Fundamentals

Before we get to probability distributions (Module 4) and statistical tests (Module 7+),
we need a shared vocabulary: what probability actually is, how to compute it, and what
happens when events depend on each other. This module is more theoretical than the
previous ones, but we'll compute every concept both "from the definition" and empirically
on our customer data.

## Table of contents
1. [What is probability?](#sec1)
2. [Basic rules: complement, union, intersection](#sec2)
3. [Conditional probability](#sec3)
4. [Independence of events](#sec4)
5. [Bayes' theorem](#sec5)
6. [The law of large numbers — a simulation](#sec6)
7. [Summary and exercises](#sec7)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. What is probability?

The **probability** of an event is a number between `[0, 1]` (or a percentage from 0% to
100%) telling you how often that event occurs. `0` means "never", `1` means "always". The
**sample space** is the set of all possible outcomes -- e.g. for rolling a six-sided die
it's `{1, 2, 3, 4, 5, 6}`. An **event** is any subset of that space, e.g. "an even number
came up" = `{2, 4, 6}`.

The simplest way to compute probability (for equally likely outcomes):

```
P(event) = number of favorable outcomes / total number of possible outcomes
```

In [ ]:
# P(rolling an even number) on a six-sided die
favorable_outcomes = 3  # {2, 4, 6}
all_outcomes = 6  # {1, 2, 3, 4, 5, 6}
p_even = favorable_outcomes / all_outcomes
print(p_even)


That's the *classical* approach (from the definition, no experiment needed). We can
also compute probability *empirically* -- from the frequency it occurs in data we already
have. For example, what's the probability that a randomly picked customer is from New
York?

In [ ]:
p_newyork = (customers["city"] == "New York").mean()
print(p_newyork)


> 📊 **mean() on a boolean condition is a trick worth remembering**
>
> `(customers["city"] == "New York")` gives a Series of `True`/`False` values. `pandas` treats `True` as `1` and `False` as `0`, so `.mean()` of that Series is exactly the fraction of rows meeting the condition -- i.e. the empirical probability. You'll see this trick repeatedly in later modules.

<a id="sec2"></a>
## 2. Basic rules: complement, union, intersection

- **Complement**: `P(not A) = 1 - P(A)`.
- **Union rule** (for events that can occur together): `P(A or B) = P(A) + P(B) -
  P(A and B)`. We subtract `P(A and B)` so we don't double-count the overlap.
- **Intersection rule** (for **independent** events): `P(A and B) = P(A) * P(B)`. More on
  independence in section 4 -- for now we're assuming it holds.

In [ ]:
p_chicago = (customers["city"] == "Chicago").mean()
p_conversion = (customers["converted"] == 1).mean()

# P(city New York OR city Chicago) -- events are MUTUALLY EXCLUSIVE (a customer has one
# city), so P(A and B) = 0 and the union rule simplifies to plain addition
p_newyork_or_chicago = p_newyork + p_chicago
print(f"P(New York or Chicago): {p_newyork_or_chicago:.3f}")

# P(city New York) * P(conversion) -- TREATING them as independent (an assumption, not a fact!)
p_newyork_and_conversion_assuming_independence = p_newyork * p_conversion
print(f"P(New York) * P(conversion) assuming independence: {p_newyork_and_conversion_assuming_independence:.3f}")


> ⚠️ **The union rule for events that do NOT exclude each other**
>
> The New York/Chicago example above is simple because a customer can't be "from New York" and "from Chicago" at the same time -- these events are **mutually exclusive** (`P(A and B) = 0`). But e.g. "customer is from New York" and "customer converted" **don't exclude each other** -- you'd need to subtract `P(New York and conversion)`, which we'll actually compute (without assuming independence) in section 4.

<a id="sec3"></a>
## 3. Conditional probability

**Conditional probability** `P(A|B)` (read: "the probability of A, given that B") is the
probability of event A once we already know B occurred. Formula:

```
P(A|B) = P(A and B) / P(B)
```

In other words: we narrow our "space of possibilities" down to only the cases where B is
true, and look at what fraction of those are also A.

In [ ]:
# P(conversion | group B) -- what's the conversion rate AMONG group B customers?
group_b = customers[customers["campaign_group"] == "B"]
p_conversion_given_b = (group_b["converted"] == 1).mean()

# the same thing, straight from the formula P(A and B) / P(B)
p_conversion_and_b = ((customers["campaign_group"] == "B") & (customers["converted"] == 1)).mean()
p_b = (customers["campaign_group"] == "B").mean()
p_conversion_given_b_from_formula = p_conversion_and_b / p_b

print(f"P(conversion | group B) directly: {p_conversion_given_b:.3f}")
print(f"P(conversion | group B) from the formula: {p_conversion_given_b_from_formula:.3f}")


Both methods give the same result -- this is exactly the same A/B effect you saw
visually in Module 2, which we'll formalize with a statistical test in Module 9
(chi-square).

<a id="sec4"></a>
## 4. Independence of events

Events A and B are **independent** when one occurring doesn't change the probability of
the other: `P(A|B) = P(A)` (equivalently: `P(A and B) = P(A) * P(B)`). If
`P(A|B) != P(A)`, the events are **dependent** -- knowing B actually tells us something
about A.

In [ ]:
p_conversion_overall = (customers["converted"] == 1).mean()
p_conversion_given_a = (customers[customers["campaign_group"] == "A"]["converted"] == 1).mean()
p_conversion_given_b = (customers[customers["campaign_group"] == "B"]["converted"] == 1).mean()

print(f"P(conversion) overall: {p_conversion_overall:.3f}")
print(f"P(conversion | group A): {p_conversion_given_a:.3f}")
print(f"P(conversion | group B): {p_conversion_given_b:.3f}")


`P(conversion | group A)` and `P(conversion | group B)` clearly differ from the overall
`P(conversion)` (and from each other) -- conversion and campaign group are **not
independent**. That's exactly the expected result: we designed the data ourselves so
group B would have a higher conversion rate.

<a id="sec5"></a>
## 5. Bayes' theorem

**Bayes' theorem** lets you "flip" a conditional probability -- from `P(B|A)` you can
compute `P(A|B)`:

```
P(A|B) = P(B|A) * P(A) / P(B)
```

Classic example: you know `P(positive test result | disease)` (the test's sensitivity),
but what you actually care about is `P(disease | positive test result)`. Let's compute an
analogous question on our own data: **given a customer who converted, what's the
probability they were in group B?**

In [ ]:
# We already know (from sections 3-4):
p_b = (customers["campaign_group"] == "B").mean()                # P(B)
p_conversion_given_b = (customers[customers["campaign_group"] == "B"]["converted"] == 1).mean()  # P(conversion | B)
p_conversion = (customers["converted"] == 1).mean()               # P(conversion)

# Bayes: P(B | conversion) = P(conversion | B) * P(B) / P(conversion)
p_b_given_conversion = p_conversion_given_b * p_b / p_conversion
print(f"P(group B | conversion): {p_b_given_conversion:.3f}")

# Verify directly from the data
converted_customers = customers[customers["converted"] == 1]
p_b_given_conversion_direct = (converted_customers["campaign_group"] == "B").mean()
print(f"Same thing, computed directly: {p_b_given_conversion_direct:.3f}")


> 📊 **Bayes' theorem is everywhere in machine learning**
>
> Spam filters, Naive Bayes classifiers, recommendation systems, medical diagnostics -- all of these fundamentally flip a known "sensitivity" (P(symptom|cause)) into the "what's really going on" we care about (P(cause|symptom)). It's one of the most important ideas in all of statistics.

<a id="sec6"></a>
## 6. The law of large numbers — a simulation

The **law of large numbers** says that the more times you repeat a random experiment, the
closer the empirical frequency gets to the true theoretical probability. Let's simulate
coin flips (P(heads) = 0.5 by definition) and watch the frequency of heads settle down as
the number of flips grows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
num_flips = 2000
flips = rng.integers(0, 2, size=num_flips)  # 0 = tails, 1 = heads

cumulative_heads_frequency = np.cumsum(flips) / np.arange(1, num_flips + 1)

plt.figure(figsize=(8, 4))
plt.plot(cumulative_heads_frequency)
plt.axhline(0.5, color="red", linestyle="--", label="True P(heads) = 0.5")
plt.xlabel("Number of flips")
plt.ylabel("Cumulative frequency of heads")
plt.title("Law of large numbers: coin flips")
plt.legend()
plt.show()


With a small number of flips the frequency "jumps around" a lot (one extra heads makes
a big percentage difference), but the more flips you take, the closer the line settles to
`0.5`. This is exactly why, in Module 5 (the Central Limit Theorem), larger samples give
more reliable estimates than small ones.

<a id="sec7"></a>
## 7. Summary and exercises

In this module we covered:
- the classical and empirical definitions of probability,
- the complement, union, and intersection rules,
- conditional probability `P(A|B)`,
- independence of events and how to check for it,
- Bayes' theorem for "flipping" a conditional probability,
- the law of large numbers as the intuition behind why larger samples are more reliable.

> 📝 **Exercise 1: Probability with two dice**
>
> Simulate (with code, not by hand) 100,000 rolls of TWO six-sided dice (`rng.integers(1, 7, size=(100000, 2))`), then compute the empirical probability that the sum equals exactly 7. Compare it to the theoretical value `6/36`.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
rng2 = np.random.default_rng(1)
two_dice_rolls = rng2.integers(1, 7, size=(100_000, 2))
total = two_dice_rolls.sum(axis=1)
p_seven = (total == 7).mean()

print(f"Empirical P(sum=7): {p_seven:.4f}")
print(f"Theoretical P(sum=7): {6/36:.4f}")
```
</details>

> 📝 **Exercise 2: Conditional satisfaction**
>
> Compute `P(satisfaction >= 7 | campaign_group == 'B')` and `P(satisfaction >= 7 | campaign_group == 'A')`. Does campaign group seem related to high satisfaction, or is it more likely just noise (the data wasn't designed that way)?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
p_high_given_b = (customers[customers["campaign_group"] == "B"]["satisfaction"] >= 7).mean()
p_high_given_a = (customers[customers["campaign_group"] == "A"]["satisfaction"] >= 7).mean()
print(p_high_given_b, p_high_given_a)
```

In our data, satisfaction depends on spend and region, NOT on campaign group -- expect these two values to be close to each other (any difference is just random noise, not a real effect).
</details>

> 📝 **Exercise 3: Independence of city and region**
>
> Check whether `P(region == 'West' | city == 'New York')` differs from `P(region == 'West')` overall. Are region and city independent in our data? (Hint: in the data-generating code, `region` is drawn completely separately from `city`.)

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
p_west_given_newyork = (customers[customers["city"] == "New York"]["region"] == "West").mean()
p_west_overall = (customers["region"] == "West").mean()
print(p_west_given_newyork, p_west_overall)
```

These should come out very close to each other (small differences are just sampling noise) -- region and city are drawn independently in the data-generating code.
</details>

> 📝 **Exercise 4: Bayes for high spenders**
>
> Bayes' theorem: compute `P(campaign_group == 'B' | monthly_spend > 300)` -- given a customer with high spend, what's the probability they're in group B? Use the Bayes formula (P(B|A) = P(A|B)*P(B)/P(A)) AND verify the result directly from the data.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
p_b = (customers["campaign_group"] == "B").mean()
p_high_given_b = (customers[customers["campaign_group"] == "B"]["monthly_spend"] > 300).mean()
p_high = (customers["monthly_spend"] > 300).mean()

p_b_given_high = p_high_given_b * p_b / p_high
print(f"From the Bayes formula: {p_b_given_high:.3f}")

direct = (customers[customers["monthly_spend"] > 300]["campaign_group"] == "B").mean()
print(f"Directly from the data: {direct:.3f}")
```

Spend doesn't depend on campaign group in our data, so expect a result close to P(B) overall (about 0.5) -- another example of no dependence.
</details>

> 🔥 **Challenge: the birthday paradox**
>
> Simulate the classic "birthday paradox": for a group of 23 random people (birthdays drawn uniformly from 365 days, `rng.integers(0, 365, size=23)`), what's the probability that at least two share the same birthday? Repeat the experiment 10,000 times (a loop or vectorization) and compute the fraction of repetitions with a match. Hint: `len(set(days)) < 23` means some two dates repeated. The result surprises most people -- check whether it comes out close to 50%.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
rng3 = np.random.default_rng(2)
num_simulations = 10_000
hits = 0

for _ in range(num_simulations):
    birthdays = rng3.integers(0, 365, size=23)
    if len(set(birthdays)) < 23:
        hits += 1

print(f"P(at least two people share a birthday): {hits / num_simulations:.3f}")
```

The theoretical value is about 0.507 -- the simulation result should land very close.
</details>

## What's next?

In **Module 4** we'll move from single events to **probability distributions** --
mathematical models describing entire families of random events: the normal, binomial,
Poisson, and uniform distributions. You'll learn to use `scipy.stats` to work with them.